In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

login(token=HF_TOKEN, add_to_git_credential=False)

info = whoami(token=HF_TOKEN)
print(f"✅ HF connecté : {info['name']}")

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Synobotix/llm-dataset.git"
REPO_NAME = "llm-dataset"

PROJECT_DIR = Path(f"/kaggle/working/{REPO_NAME}")

if PROJECT_DIR.exists():
    print("📁 Repo déjà cloné — pull")
    subprocess.run(["git", "pull"], cwd=PROJECT_DIR, check=True)
else:
    print("📥 Clonage...")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print(f"✅ CWD : {os.getcwd()}")

In [ ]:
import os
import subprocess

print("📦 Installation de Poetry...")

subprocess.run(
    "curl -sSL https://install.python-poetry.org | python3 -",
    shell=True,
    check=True,
)

os.environ["PATH"] = (
    f"{os.path.expanduser('~')}/.local/bin:"
    + os.environ["PATH"]
)

result = subprocess.run(
    ["poetry", "--version"],
    capture_output=True,
    text=True,
)
print(result.stdout)

In [ ]:
import subprocess

print("📦 Installation de Python 3.11...")

subprocess.run(
    ["apt-get", "update"],
    check=True,
)

subprocess.run(
    ["apt-get", "install", "-y",
     "python3.11", "python3.11-venv", "python3.11-dev"],
    check=True,
)

result = subprocess.run(
    ["python3.11", "--version"],
    capture_output=True,
    text=True,
)
print(f"✅ Python 3.11 : {result.stdout.strip()}")

In [ ]:
import os
import subprocess

os.chdir("/kaggle/working/llm-dataset")

# ============================================================
# 1. Configurer Poetry pour utiliser Python 3.11
# ============================================================

print("🔧 Configuration de Poetry pour Python 3.11...")

subprocess.run(
    ["poetry", "env", "use", "python3.11"],
    check=True,
)

# ============================================================
# 2. Installer les dépendances
# ============================================================

print("\n📦 Installation des dépendances Poetry...")

subprocess.run(
    ["poetry", "install", "--no-root"],
    check=True,
)

result = subprocess.run(
    ["poetry", "run", "python", "--version"],
    capture_output=True,
    text=True,
)
print(f"✅ Python dans Poetry : {result.stdout.strip()}")

# ============================================================
# 3. Désinstaller torch CPU
# ============================================================

subprocess.run(
    ["poetry", "run", "pip", "uninstall", "-y",
     "torch", "torchvision", "torchaudio"],
    check=False,
)

# ============================================================
# 4. Installer torch CUDA 12.1
# ============================================================

print("\n📦 Installation torch CUDA 12.1...")

result = subprocess.run(
    ["poetry", "run", "pip", "install",
     "torch==2.4.1", "torchvision==0.19.1", "torchaudio==2.4.1",
     "--index-url", "https://download.pytorch.org/whl/cu121"],
    capture_output=True,
    text=True,
)

if result.returncode != 0:
    print("❌ Échec CUDA")
    print("STDOUT:", result.stdout[-1000:])
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError("Installation torch CUDA échouée")

print("✅ torch CUDA installé")

# ============================================================
# 5. Vérification finale
# ============================================================

result = subprocess.run(
    ["poetry", "run", "python", "-c",
     "import torch; "
     "print('torch', torch.__version__); "
     "print('cuda', torch.cuda.is_available()); "
     "print('gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')"],
    capture_output=True,
    text=True,
)

print("\n" + "=" * 70)
print("RÉSULTAT FINAL")
print("=" * 70)
print(result.stdout)

In [ ]:
import os

os.chdir("/kaggle/working/llm-dataset")

# PYTHONPATH
os.environ["PYTHONPATH"] = (
    "/kaggle/working/llm-dataset/src:"
    "/kaggle/working/llm-dataset"
)

# HF_TOKEN dans l'environnement
os.environ["HF_TOKEN"] = HF_TOKEN

print(f"✅ PYTHONPATH configuré")
print(f"   {os.environ['PYTHONPATH']}")

In [ ]:
import subprocess

os.chdir("/kaggle/working/llm-dataset")

result = subprocess.run(
    ["poetry", "run", "python", "-c",
     "import torch; "
     "print('torch :', torch.__version__); "
     "print('cuda  :', torch.cuda.is_available()); "
     "print('count :', torch.cuda.device_count()); "
     "if torch.cuda.is_available(): "
     "    for i in range(torch.cuda.device_count()): "
     "        print(f'gpu_{i} :', torch.cuda.get_device_name(i))"],
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.stderr:
    print("STDERR:", result.stderr[-500:])

In [ ]:
import subprocess
import os

os.chdir("/kaggle/working/llm-dataset")

# Lancer le monitor en arrière-plan
log_file = open("/kaggle/working/gpu_monitor.log", "w")

monitor = subprocess.Popen(
    ["poetry", "run", "python", "-u", "-m",
     "scripts.bitnet.heartbeat.gpu_monitor"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
    start_new_session=True,
)

print(f"✅ GPU Monitor lancé (PID={monitor.pid})")

In [ ]:
import torch

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        allocated = torch.cuda.memory_allocated(i) / 1e9
        reserved = torch.cuda.memory_reserved(i) / 1e9
        total = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(
            f"GPU {i} ({torch.cuda.get_device_name(i)}): "
            f"allocated={allocated:.2f} Go, "
            f"reserved={reserved:.2f} Go, "
            f"total={total:.2f} Go"
        )
else:
    print("Pas de GPU disponible")

In [ ]:
import subprocess
import os

os.chdir("/kaggle/working/llm-dataset")

STEPS = [
    "scripts.bitnet.reset_data.delete_all_data_json_jsonl_pt_local",
    "scripts.bitnet.reset_data.download_data_json_jsonl_md",
    "scripts.bitnet.train_bitnet",
    "scripts.bitnet.analyse_monitoring",
]

for step in STEPS:
    print(f"\n{'=' * 70}")
    print(f"▶ {step}")
    print("=" * 70)
    
    result = subprocess.run(
        ["poetry", "run", "python", "-m", step],
        env=os.environ.copy(),
        text=True,
    )
    
    if result.returncode != 0:
        print(f"\n❌ Échec : {step}")
        break
    
    print(f"\n✅ {step} OK")